In [ ]:
import numpy as np

# -------------------------
# Activations + derivatives
# -------------------------
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def dsigmoid_from_sigmoid(s):
    # s = sigmoid(x)
    return s * (1.0 - s)

def dtanh_from_tanh(t):
    # t = tanh(x)
    return 1.0 - t * t

# -------------------------
# GRU from scratch (NumPy)
# many-to-one regression
# -------------------------
class GRUFromScratch:
    def __init__(self, input_size, hidden_size, out_size=1, seed=42):
        rng = np.random.default_rng(seed)
        D, H, O = input_size, hidden_size, out_size

        # Xavier-ish init
        def init_w(fan_in, fan_out):
            limit = np.sqrt(6.0 / (fan_in + fan_out))
            return rng.uniform(-limit, limit, size=(fan_in, fan_out))

        # Gates: z, r, h~
        self.W_xz = init_w(D, H)
        self.W_hz = init_w(H, H)
        self.b_z  = np.zeros((H,))

        self.W_xr = init_w(D, H)
        self.W_hr = init_w(H, H)
        self.b_r  = np.zeros((H,))

        self.W_xh = init_w(D, H)
        self.W_hh = init_w(H, H)
        self.b_h  = np.zeros((H,))

        # Output layer (many-to-one)
        self.W_y = init_w(H, O)
        self.b_y = np.zeros((O,))

        # grads (same shapes)
        self.zero_grads()

    def zero_grads(self):
        D, H = self.W_xz.shape
        O = self.W_y.shape[1]

        self.dW_xz = np.zeros_like(self.W_xz)
        self.dW_hz = np.zeros_like(self.W_hz)
        self.db_z  = np.zeros_like(self.b_z)

        self.dW_xr = np.zeros_like(self.W_xr)
        self.dW_hr = np.zeros_like(self.W_hr)
        self.db_r  = np.zeros_like(self.b_r)

        self.dW_xh = np.zeros_like(self.W_xh)
        self.dW_hh = np.zeros_like(self.W_hh)
        self.db_h  = np.zeros_like(self.b_h)

        self.dW_y = np.zeros_like(self.W_y)
        self.db_y = np.zeros_like(self.b_y)

    # --------
    # Forward
    # --------
    def forward(self, X, h0=None):
        """
        X: (B, T, D)
        returns: y_hat (B, O), cache for backward
        """
        B, T, D = X.shape
        H = self.W_hz.shape[0]

        if h0 is None:
            h_prev = np.zeros((B, H))
        else:
            h_prev = h0

        # store per-time caches for BPTT
        cache = {
            "X": X,
            "h": [h_prev],   # h[0] = h0, then h[1..T]
            "z": [],
            "r": [],
            "h_tilde": [],
            "a_z": [],
            "a_r": [],
            "a_h": [],
        }

        for t in range(T):
            x_t = X[:, t, :]                       # (B, D)

            a_z = x_t @ self.W_xz + h_prev @ self.W_hz + self.b_z
            z   = sigmoid(a_z)

            a_r = x_t @ self.W_xr + h_prev @ self.W_hr + self.b_r
            r   = sigmoid(a_r)

            a_h = x_t @ self.W_xh + (r * h_prev) @ self.W_hh + self.b_h
            h_tilde = np.tanh(a_h)

            h_t = (1.0 - z) * h_prev + z * h_tilde

            cache["a_z"].append(a_z); cache["z"].append(z)
            cache["a_r"].append(a_r); cache["r"].append(r)
            cache["a_h"].append(a_h); cache["h_tilde"].append(h_tilde)
            cache["h"].append(h_t)

            h_prev = h_t

        h_T = cache["h"][-1]                       # (B, H)
        y_hat = h_T @ self.W_y + self.b_y          # (B, O)
        cache["y_hat"] = y_hat
        return y_hat, cache

    # --------------------
    # Loss: MSE (mean)
    # --------------------
    @staticmethod
    def mse_loss(y_hat, y_true):
        # y_hat, y_true: (B, O)
        diff = y_hat - y_true
        return np.mean(diff * diff)

    @staticmethod
    def dmse_dyhat(y_hat, y_true):
        # d/dyhat mean((yhat-y)^2) = 2*(yhat-y)/(B*O)
        B, O = y_hat.shape
        return (2.0 * (y_hat - y_true)) / (B * O)

    # ---------
    # Backward
    # ---------
    def backward(self, cache, y_true):
        """
        cache from forward, y_true (B, O)
        computes grads into self.d*
        returns: None
        """
        self.zero_grads()

        X = cache["X"]
        B, T, D = X.shape
        H = self.W_hz.shape[0]

        y_hat = cache["y_hat"]
        dy = self.dmse_dyhat(y_hat, y_true)        # (B, O)

        # Output layer grads: y_hat = h_T W_y + b_y
        h_T = cache["h"][-1]                       # (B, H)
        self.dW_y += h_T.T @ dy                    # (H, O)
        self.db_y += np.sum(dy, axis=0)            # (O,)

        # gradient flowing into h_T
        dh_next = dy @ self.W_y.T                  # (B, H)

        # BPTT through time t = T-1 ... 0
        for t in reversed(range(T)):
            x_t     = X[:, t, :]                   # (B, D)
            h_prev  = cache["h"][t]                # (B, H)  (h_{t})
            h_t     = cache["h"][t+1]              # (B, H)  (h_{t+1})
            z       = cache["z"][t]
            r       = cache["r"][t]
            h_tilde = cache["h_tilde"][t]

            # h_t = (1-z)*h_prev + z*h_tilde
            dh = dh_next                           # current grad wrt h_t

            # partials
            # w.r.t z: d h_t / d z = -h_prev + h_tilde
            dz = dh * (-h_prev + h_tilde)          # (B, H)

            # w.r.t h_tilde: d h_t / d h_tilde = z
            dh_tilde = dh * z                       # (B, H)

            # w.r.t h_prev direct path: d h_t / d h_prev = (1-z)
            dh_prev = dh * (1.0 - z)                # (B, H)

            # h_tilde = tanh(a_h)
            da_h = dh_tilde * dtanh_from_tanh(h_tilde)  # (B, H)

            # a_h = x_t W_xh + (r*h_prev) W_hh + b_h
            self.dW_xh += x_t.T @ da_h
            self.dW_hh += (r * h_prev).T @ da_h
            self.db_h  += np.sum(da_h, axis=0)

            # gradient wrt (r*h_prev)
            d_rhprev = da_h @ self.W_hh.T           # (B, H)

            # split d_rhprev to r and h_prev
            dr = d_rhprev * h_prev                  # (B, H)
            dh_prev += d_rhprev * r                 # add to dh_prev

            # r = sigmoid(a_r)
            da_r = dr * dsigmoid_from_sigmoid(r)    # (B, H)

            # a_r = x_t W_xr + h_prev W_hr + b_r
            self.dW_xr += x_t.T @ da_r
            self.dW_hr += h_prev.T @ da_r
            self.db_r  += np.sum(da_r, axis=0)

            # contribute to dh_prev from a_r path
            dh_prev += da_r @ self.W_hr.T           # (B, H)

            # z = sigmoid(a_z)
            da_z = dz * dsigmoid_from_sigmoid(z)    # (B, H)

            # a_z = x_t W_xz + h_prev W_hz + b_z
            self.dW_xz += x_t.T @ da_z
            self.dW_hz += h_prev.T @ da_z
            self.db_z  += np.sum(da_z, axis=0)

            # contribute to dh_prev from a_z path
            dh_prev += da_z @ self.W_hz.T           # (B, H)

            # now pass to previous time step
            dh_next = dh_prev

    # ----------------
    # Optimizer: SGD
    # ----------------
    def step_sgd(self, lr=1e-2, grad_clip=None):
        # optional global norm clip (simple)
        if grad_clip is not None:
            # collect all grads into one vector to compute norm
            grads = [
                self.dW_xz, self.dW_hz, self.db_z,
                self.dW_xr, self.dW_hr, self.db_r,
                self.dW_xh, self.dW_hh, self.db_h,
                self.dW_y,  self.db_y
            ]
            norm = np.sqrt(sum(np.sum(g*g) for g in grads))
            if norm > grad_clip:
                scale = grad_clip / (norm + 1e-12)
                for g in grads:
                    g *= scale

        # apply update
        self.W_xz -= lr * self.dW_xz; self.W_hz -= lr * self.dW_hz; self.b_z -= lr * self.db_z
        self.W_xr -= lr * self.dW_xr; self.W_hr -= lr * self.dW_hr; self.b_r -= lr * self.db_r
        self.W_xh -= lr * self.dW_xh; self.W_hh -= lr * self.dW_hh; self.b_h -= lr * self.db_h
        self.W_y  -= lr * self.dW_y;  self.b_y  -= lr * self.db_y


# -------------------------
# Demo training (toy task)
# -------------------------
if __name__ == "__main__":
    np.random.seed(0)

    B, T, D = 32, 12, 4
    H, O = 32, 1

    model = GRUFromScratch(input_size=D, hidden_size=H, out_size=O)

    # toy data: y = sum of all x + noise
    X = np.random.randn(B, T, D)
    y = X.sum(axis=(1, 2), keepdims=True) + 0.1*np.random.randn(B, 1)

    for ep in range(1, 301):
        y_hat, cache = model.forward(X)
        loss = model.mse_loss(y_hat, y)

        model.backward(cache, y)
        model.step_sgd(lr=0.05, grad_clip=5.0)

        if ep % 50 == 0:
            print(f"epoch {ep:3d} | loss {loss:.6f}")